In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/cybersecurity-intrusion-detection-dataset/httpswww.kaggle.comdatasetsdnkumarscybersecurity-intrusion-detection-dataset.csv
/kaggle/input/improved-cicids2017-and-csecicids2018/CICIDS2017_improved/tuesday.csv
/kaggle/input/improved-cicids2017-and-csecicids2018/CICIDS2017_improved/monday.csv
/kaggle/input/improved-cicids2017-and-csecicids2018/CICIDS2017_improved/friday.csv
/kaggle/input/improved-cicids2017-and-csecicids2018/CICIDS2017_improved/wednesday.csv
/kaggle/input/improved-cicids2017-and-csecicids2018/CICIDS2017_improved/thursday.csv
/kaggle/input/improved-cicids2017-and-csecicids2018/CSECICIDS2018_improved/Friday-02-03-2018.csv
/kaggle/input/improved-cicids2017-and-csecicids2018/CSECICIDS2018_improved/Thursday-01-03-2018.csv
/kaggle/input/improved-cicids2017-and-csecicids2018/CSECICIDS2018_improved/Thursday-22-02-2018.csv
/kaggle/input/improved-cicids2017-and-csecicids2018/CSECICIDS2018_improved/Wednesday-21-02-2018.csv
/kaggle/input/improved-cicids2017-and-csecicids

In [2]:
!dir /kaggle/input/improved-cicids2017-and-csecicids2018/CSECICIDS2018_improved

Friday-02-03-2018.csv	 Thursday-15-02-2018.csv   Wednesday-21-02-2018.csv
Friday-16-02-2018.csv	 Thursday-22-02-2018.csv   Wednesday-28-02-2018.csv
Friday-23-02-2018.csv	 Tuesday-20-02-2018.csv
Thursday-01-03-2018.csv  Wednesday-14-02-2018.csv


In [3]:
!pip install pytorch-tabnet 
!pip install dask[complete]
!pip install distributed 
!pip install pytorch-tabnet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.3 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 120.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.6/419.6 kB 29.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 7.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new

In [1]:
# import pandas as pd
import dask.dataframe as dd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, RocCurveDisplay, roc_auc_score, precision_score, recall_score, f1_score
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

In [2]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cpu'

# Phase 1: Business Understanding

Business Understanding forms the foundation of any successful data mining project and is crucial for defining clear objectives and success criteria. In the context of network intrusion detection using TabNet.

# Phase 2: Data Understanding

Data Understanding involves comprehensive exploration and analysis of the network flow dataset to gain insights into its structure, quality, and characteristics.

In [4]:
# Example with sample data
df = dd.read_csv('data/Friday-02-03-2018.csv', assume_missing=True)

In [3]:
model_columns = ['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet',
       'Total Length of Fwd Packet', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Mean', 'Fwd IAT Std', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags',
       'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Min',
       'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'CWR Flag Count',
       'ECE Flag Count', 'Down/Up Ratio', 'Fwd Bytes/Bulk Avg',
       'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg', 'Bwd Bytes/Bulk Avg',
       'Bwd Bulk Rate Avg', 'Subflow Fwd Packets', 'Subflow Bwd Packets',
       'FWD Init Win Bytes', 'Bwd Init Win Bytes', 'Fwd Act Data Pkts',
       'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max',
       'Active Min', 'Idle Std', 'ICMP Code', 'ICMP Type',
       'Total TCP Flow Time', 'Attempted Category']

In [ ]:
X

In [7]:
dfc = df.compute()

In [8]:
dfc.head(10)

,id,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,ICMP Code,ICMP Type,Total TCP Flow Time,Label,Attempted Category
0,1.0,192.168.10.50-192.168.10.3-56108-3268-6,192.168.10.50,56108.0,192.168.10.3,3268.0,6.0,2017-07-07 11:59:50.315195,112740690.0,32.0,...,343.0,1.610540e+07,4.988048e+05,16399772.0,15375229.0,-1.0,-1.0,112740690.0,BENIGN,-1.0
1,2.0,192.168.10.50-192.168.10.3-42144-389-6,192.168.10.50,42144.0,192.168.10.3,389.0,6.0,2017-07-07 11:59:50.316273,112740560.0,32.0,...,285.0,1.610543e+07,4.987937e+05,16399782.0,15375263.0,-1.0,-1.0,112740560.0,BENIGN,-1.0
2,3.0,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0.0,8.0.6.4,0.0,0.0,2017-07-07 12:00:31.388567,113757377.0,545.0,...,19.0,1.221036e+07,6.935824e+06,20757030.0,5504997.0,-1.0,-1.0,0.0,BENIGN,-1.0
3,4.0,192.168.10.25-224.0.0.251-5353-5353-17,192.168.10.25,5353.0,224.0.0.251,5353.0,17.0,2017-07-07 12:00:42.903850,91997219.0,388.0,...,16.0,1.319764e+07,5.826905e+06,19776791.0,5817470.0,-1.0,-1.0,0.0,BENIGN,-1.0
4,5.0,192.168.10.25-17.253.14.125-123-123-17,192.168.10.25,123.0,17.253.14.125,123.0,17.0,2017-07-07 12:00:42.430758,66966070.0,6.0,...,1968172.0,6.497443e+07,0.000000e+00,64974431.0,64974431.0,-1.0,-1.0,0.0,BENIGN,-1.0
5,6.0,192.168.10.5-192.168.10.3-123-123-17,192.168.10.5,123.0,192.168.10.3,123.0,17.0,2017-07-07 12:00:53.296592,66137889.0,6.0,...,1500785.0,6.463703e+07,0.000000e+00,64637028.0,64637028.0,-1.0,-1.0,0.0,BENIGN,-1.0
6,7.0,192.168.10.5-192.168.10.3-49159-445-6,192.168.10.5,49159.0,192.168.10.3,445.0,6.0,2017-07-07 12:00:50.241989,106176490.0,124.0,...,43.0,2.102008e+07,1.237816e+07,30004236.0,5349458.0,-1.0,-1.0,106176490.0,BENIGN,-1.0
7,8.0,192.168.10.5-134.170.115.55-49176-443-6,192.168.10.5,49176.0,134.170.115.55,443.0,6.0,2017-07-07 12:00:55.280312,79593267.0,10.0,...,83707.0,3.945696e+07,2.068116e+07,54080751.0,24833175.0,-1.0,-1.0,79593267.0,BENIGN,-1.0
8,9.0,192.168.10.9-192.168.10.3-123-123-17,192.168.10.9,123.0,192.168.10.3,123.0,17.0,2017-07-07 12:01:07.159677,64318195.0,4.0,...,101.0,6.431801e+07,0.000000e+00,64318007.0,64318007.0,-1.0,-1.0,0.0,BENIGN,-1.0
9,10.0,192.168.10.14-192.168.10.3-49414-445-6,192.168.10.14,49414.0,192.168.10.3,445.0,6.0,2017-07-07 12:00:48.901484,116933759.0,206.0,...,36.0,1.908480e+07,1.204026e+07,30000694.0,5323604.0,-1.0,-1.0,116933759.0,BENIGN,-1.0


In [9]:
dfc.shape

(2099976, 91)

In [10]:
# Target variable analysis
print(f"\nTarget Variable Analysis:")
print(f"Label column: 'Label'")
print(f"Unique classes: {dfc['Label'].nunique()}")
print(f"\nClass distribution:")
class_distribution = dfc['Label'].value_counts()
for class_name, count in class_distribution.items():
    percentage = (count / len(dfc)) * 100
    print(f"  {class_name}: {count} ({percentage:.5f}%)")


Target Variable Analysis:
Label column: 'Label'
Unique classes: 27

Class distribution:
  BENIGN: 1582566 (75.36115%)
  Portscan: 159066 (7.57466%)
  DoS Hulk: 158468 (7.54618%)
  DDoS: 95144 (4.53072%)
  Infiltration - Portscan: 71767 (3.41752%)
  DoS GoldenEye: 7567 (0.36034%)
  Botnet - Attempted: 4067 (0.19367%)
  FTP-Patator: 3972 (0.18915%)
  DoS Slowloris: 3859 (0.18376%)
  DoS Slowhttptest - Attempted: 3368 (0.16038%)
  SSH-Patator: 2961 (0.14100%)
  DoS Slowloris - Attempted: 1847 (0.08795%)
  DoS Slowhttptest: 1740 (0.08286%)
  Web Attack - Brute Force - Attempted: 1292 (0.06152%)
  Botnet: 736 (0.03505%)
  Web Attack - XSS - Attempted: 655 (0.03119%)
  DoS Hulk - Attempted: 581 (0.02767%)
  DoS GoldenEye - Attempted: 80 (0.00381%)
  Web Attack - Brute Force: 73 (0.00348%)
  Infiltration - Attempted: 45 (0.00214%)
  Infiltration: 36 (0.00171%)
  SSH-Patator - Attempted: 27 (0.00129%)
  Web Attack - XSS: 18 (0.00086%)
  Web Attack - SQL Injection: 13 (0.00062%)
  FTP-Patator 

In [11]:
# Data overview
print(f"Dataset Shape: {dfc.shape}")
print(f"Features: {dfc.shape[1] - 1}")  # -1 for label column
print(f"Samples: {dfc.shape[0]}")

Dataset Shape: (2099976, 91)
Features: 90
Samples: 2099976


In [12]:
# Data types analysis
print(f"\nData Types:")
print(dfc.dtypes.value_counts())



Data Types:
float64            86
string[pyarrow]     5
Name: count, dtype: int64


In [13]:
# Target variable analysis
print(f"\nTarget Variable Analysis:")
print(f"Label column: 'Label'")
print(f"Unique classes: {dfc['Label'].nunique()}")
print(f"\nClass distribution:")
class_distribution = dfc['Label'].value_counts()
for class_name, count in class_distribution.items():
    percentage = (count / len(dfc)) * 100
    print(f"  {class_name}: {count} ({percentage:.5f}%)")


Target Variable Analysis:
Label column: 'Label'
Unique classes: 27

Class distribution:
  BENIGN: 1582566 (75.36115%)
  Portscan: 159066 (7.57466%)
  DoS Hulk: 158468 (7.54618%)
  DDoS: 95144 (4.53072%)
  Infiltration - Portscan: 71767 (3.41752%)
  DoS GoldenEye: 7567 (0.36034%)
  Botnet - Attempted: 4067 (0.19367%)
  FTP-Patator: 3972 (0.18915%)
  DoS Slowloris: 3859 (0.18376%)
  DoS Slowhttptest - Attempted: 3368 (0.16038%)
  SSH-Patator: 2961 (0.14100%)
  DoS Slowloris - Attempted: 1847 (0.08795%)
  DoS Slowhttptest: 1740 (0.08286%)
  Web Attack - Brute Force - Attempted: 1292 (0.06152%)
  Botnet: 736 (0.03505%)
  Web Attack - XSS - Attempted: 655 (0.03119%)
  DoS Hulk - Attempted: 581 (0.02767%)
  DoS GoldenEye - Attempted: 80 (0.00381%)
  Web Attack - Brute Force: 73 (0.00348%)
  Infiltration - Attempted: 45 (0.00214%)
  Infiltration: 36 (0.00171%)
  SSH-Patator - Attempted: 27 (0.00129%)
  Web Attack - XSS: 18 (0.00086%)
  Web Attack - SQL Injection: 13 (0.00062%)
  FTP-Patator 

In [14]:
# Feature categories (based on network flow analysis)
feature_categories = {
    "Packet_Features": [col for col in dfc.columns if 'Pkt' in col],
    "Flow_Features": [col for col in dfc.columns if 'Flow' in col],
    "Forward_Features": [col for col in dfc.columns if 'Fwd' in col],
    "Backward_Features": [col for col in dfc.columns if 'Bwd' in col],
    "Flag_Features": [col for col in dfc.columns if 'Flag' in col],
    "Timing_Features": [col for col in dfc.columns if 'IAT' in col or 'Duration' in col]
}
print(f"\nFeature Categories:")
for category, features in feature_categories.items():
    print(f"  {category}: {len(features)} features")


Feature Categories:
  Packet_Features: 1 features
  Flow_Features: 9 features
  Forward_Features: 24 features
  Backward_Features: 23 features
  Flag_Features: 14 features
  Timing_Features: 15 features


## Data Quality Assessment


In [15]:
# Missing values
missing_values = dfc.isnull().sum().sum()
print(f"Missing values: {missing_values}")


Missing values: 0


In [16]:
# Infinite values
numeric_cols = dfc.select_dtypes(include=[np.number]).columns
infinite_values = np.isinf(dfc[numeric_cols]).sum().sum()
print(f"Infinite values: {infinite_values}")

Infinite values: 10


In [17]:
# Duplicate rows
duplicates = dfc.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

Duplicate rows: 0


In [18]:
# Feature statistics
print(f"\nFeature Statistics:")
print(dfc.describe())


Feature Statistics:


/usr/local/lib/python3.10/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/usr/local/lib/python3.10/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


                 id      Src Port      Dst Port      Protocol  Flow Duration  \
count  2.099976e+06  2.099976e+06  2.099976e+06  2.099976e+06   2.099976e+06   
mean   2.189098e+05  4.848723e+04  1.249651e+03  1.122765e+01   1.244028e+07   
std    1.362580e+05  1.595439e+04  5.820514e+03  5.502014e+00   3.103048e+07   
min    1.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   0.000000e+00   
25%    1.049990e+05  4.275700e+04  5.300000e+01  6.000000e+00   2.480000e+02   
50%    2.099980e+05  5.348000e+04  8.000000e+01  6.000000e+00   6.182600e+04   
75%    3.149970e+05  6.023200e+04  4.430000e+02  1.700000e+01   4.388213e+06   
max    5.475570e+05  6.553500e+04  6.552900e+04  1.700000e+01   1.200000e+08   

       Total Fwd Packet  Total Bwd packets  Total Length of Fwd Packet  \
count      2.099976e+06       2.099976e+06                2.099976e+06   
mean       1.252090e+01       1.420793e+01                5.704216e+02   
std        8.703759e+02       1.157985e+03               

In [19]:
# Check class distribution
print("\nClass distribution:")
print(dfc['Label'].value_counts())


Class distribution:
Label
BENIGN                                    1582566
Portscan                                   159066
DoS Hulk                                   158468
DDoS                                        95144
Infiltration - Portscan                     71767
DoS GoldenEye                                7567
Botnet - Attempted                           4067
FTP-Patator                                  3972
DoS Slowloris                                3859
DoS Slowhttptest - Attempted                 3368
SSH-Patator                                  2961
DoS Slowloris - Attempted                    1847
DoS Slowhttptest                             1740
Web Attack - Brute Force - Attempted         1292
Botnet                                        736
Web Attack - XSS - Attempted                  655
DoS Hulk - Attempted                          581
DoS GoldenEye - Attempted                      80
Web Attack - Brute Force                       73
Infiltration - Attempte

# Phase 3: Data Preparation

## 0. Drop unecessary columns

In [20]:
data_to_drop = [col for col in dfc.columns if 'id' in col or 'ID' in col or 'IP' in col]
data_to_drop.append('Timestamp')

In [21]:
data_to_drop

['id', 'Flow ID', 'Src IP', 'Dst IP', 'Timestamp']

In [22]:
dfc = dfc.drop(data_to_drop, axis=1)

## 1. Handling missing values;

In [23]:
# Make a copy of the dataset
data = dfc.copy()
# delete df for more ram
del dfc

# Handle missing values for numeric columns
numeric_cols = data.select_dtypes(include=[np.number]).columns
print(f"Numeric columns: {len(numeric_cols)}")
# Fill missing values with median
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].median())

Numeric columns: 85


## 2. Handle infinite values

In [24]:
data = data.replace([np.inf, -np.inf], np.nan)
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].median())

## 3. Remove duplicates


In [25]:
data = data.drop_duplicates()

##  4. Feature selection


In [26]:
print("4. Feature analysis...")
# Correlation analysis
correlation_matrix = data[numeric_cols].corr().abs()
high_corr_features = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.95:
            high_corr_features.append((correlation_matrix.columns[i], correlation_matrix.columns[j]))

print(f"Highly correlated feature pairs (>0.95): {len(high_corr_features)}")



4. Feature analysis...
Highly correlated feature pairs (>0.95): 54


In [27]:
# Create an upper triangle mask (to avoid duplicate pairs)
upper = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))

# Identify columns to drop (any feature correlated > 0.95 with another)
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

print("Highly correlated columns to drop (|corr| > 0.95):")
print(to_drop)

# Drop them from your dataset
data_uncorr = data.drop(columns=to_drop)

print(f"Shape before: {data.shape}")
print(f"Shape after:  {data_uncorr.shape}")


Highly correlated columns to drop (|corr| > 0.95):
['Total Bwd packets', 'Total Length of Bwd Packet', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Fwd IAT Total', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Packets/s', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'Average Packet Size', 'Fwd Segment Size Avg', 'Bwd Segment Size Avg', 'Bwd Packet/Bulk Avg', 'Subflow Fwd Bytes', 'Subflow Bwd Bytes', 'Idle Mean', 'Idle Max', 'Idle Min']
Shape before: (2084721, 86)
Shape after:  (2084721, 59)


## 5. Data preparation summary

In [28]:
print(f"\nData Preparation Summary:")
print(f"Cleaned samples: {data_uncorr.shape[0]}")
print(f"Features: {data_uncorr.shape[1] - 1}")


Data Preparation Summary:
Cleaned samples: 2084721
Features: 58


In [29]:
data_uncorr.head(10)

,Src Port,Dst Port,Protocol,Flow Duration,Total Fwd Packet,Total Length of Fwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Mean,Active Std,Active Max,Active Min,Idle Std,ICMP Code,ICMP Type,Total TCP Flow Time,Label,Attempted Category
0,56108.0,3268.0,6.0,112740690.0,32.0,6448.0,403.0,0.0,201.500000,204.724205,...,3.594286e+02,1.199802e+01,380.0,343.0,4.988048e+05,-1.0,-1.0,112740690.0,BENIGN,-1.0
1,42144.0,389.0,6.0,112740560.0,32.0,6448.0,403.0,0.0,201.500000,204.724205,...,3.202857e+02,1.574499e+01,330.0,285.0,4.987937e+05,-1.0,-1.0,112740560.0,BENIGN,-1.0
2,0.0,0.0,0.0,113757377.0,545.0,0.0,0.0,0.0,0.000000,0.000000,...,9.361829e+06,7.324646e+06,18851791.0,19.0,6.935824e+06,-1.0,-1.0,0.0,BENIGN,-1.0
3,5353.0,5353.0,17.0,91997219.0,388.0,37151.0,227.0,37.0,95.750000,55.785320,...,9.801664e+06,1.152782e+07,24721964.0,16.0,5.826905e+06,-1.0,-1.0,0.0,BENIGN,-1.0
4,123.0,123.0,17.0,66966070.0,6.0,288.0,48.0,48.0,48.000000,0.000000,...,1.968172e+06,0.000000e+00,1968172.0,1968172.0,0.000000e+00,-1.0,-1.0,0.0,BENIGN,-1.0
5,123.0,123.0,17.0,66137889.0,6.0,288.0,48.0,48.0,48.000000,0.000000,...,1.500785e+06,0.000000e+00,1500785.0,1500785.0,0.000000e+00,-1.0,-1.0,0.0,BENIGN,-1.0
6,49159.0,445.0,6.0,106176490.0,124.0,34190.0,3072.0,0.0,275.725806,457.272834,...,2.152050e+05,4.253848e+05,974787.0,43.0,1.237816e+07,-1.0,-1.0,106176490.0,BENIGN,-1.0
7,49176.0,443.0,6.0,79593267.0,10.0,1602.0,421.0,0.0,160.200000,187.822493,...,1.115080e+05,3.931655e+04,139309.0,83707.0,2.068116e+07,-1.0,-1.0,79593267.0,BENIGN,-1.0
8,123.0,123.0,17.0,64318195.0,4.0,192.0,48.0,48.0,48.000000,0.000000,...,1.010000e+02,0.000000e+00,101.0,101.0,0.000000e+00,-1.0,-1.0,0.0,BENIGN,-1.0
9,49414.0,445.0,6.0,116933759.0,206.0,42254.0,3085.0,0.0,205.116505,365.897581,...,4.041533e+05,8.900203e+05,2219566.0,36.0,1.204026e+07,-1.0,-1.0,116933759.0,BENIGN,-1.0


## Feature Encoding and Splitting


In [30]:
# Separate features and target
X = data_uncorr.drop('Label', axis=1)
y = data_uncorr['Label']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {X.columns.tolist()}")

Features shape: (2084721, 58)
Target shape: (2084721,)
Feature columns: ['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet', 'Total Length of Fwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Mean', 'Fwd IAT Std', 'Bwd IAT Mean', 'Bwd IAT Std', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Min', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'CWR Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg', 'Bwd Bytes/Bulk Avg', 'Bwd Bulk Rate Avg', 'Subflow Fwd Packets', 'Subflow Bwd Packets', 'FWD Init Win Bytes', 'Bwd Init Win Bytes', 'Fwd Act Data Pkts', 'Fw

In [31]:
# Check for categorical columns
categorical_columns = X.select_dtypes(include=['object']).columns
print(f"Categorical columns: {categorical_columns.tolist()}")

# Encode categorical variables if any
for col in categorical_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    print(f"Encoded {col}")

# Encode target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\nClasses: {label_encoder.classes_}")
print(f"Number of classes: {len(label_encoder.classes_)}")

Categorical columns: []

Classes: ['BENIGN' 'Botnet' 'Botnet - Attempted' 'DDoS' 'DoS GoldenEye'
 'DoS GoldenEye - Attempted' 'DoS Hulk' 'DoS Hulk - Attempted'
 'DoS Slowhttptest' 'DoS Slowhttptest - Attempted' 'DoS Slowloris'
 'DoS Slowloris - Attempted' 'FTP-Patator' 'FTP-Patator - Attempted'
 'Heartbleed' 'Infiltration' 'Infiltration - Attempted'
 'Infiltration - Portscan' 'Portscan' 'SSH-Patator'
 'SSH-Patator - Attempted' 'Web Attack - Brute Force'
 'Web Attack - Brute Force - Attempted' 'Web Attack - SQL Injection'
 'Web Attack - SQL Injection - Attempted' 'Web Attack - XSS'
 'Web Attack - XSS - Attempted']
Number of classes: 27


In [32]:
# Display class distribution after encoding
unique, counts = np.unique(y_encoded, return_counts=True)
for i, (class_idx, count) in enumerate(zip(unique, counts)):
    print(f"Class {class_idx} ({label_encoder.classes_[class_idx]}): {count} samples")


Class 0 (BENIGN): 1570468 samples
Class 1 (Botnet): 736 samples
Class 2 (Botnet - Attempted): 4064 samples
Class 3 (DDoS): 95144 samples
Class 4 (DoS GoldenEye): 7567 samples
Class 5 (DoS GoldenEye - Attempted): 80 samples
Class 6 (DoS Hulk): 158468 samples
Class 7 (DoS Hulk - Attempted): 581 samples
Class 8 (DoS Slowhttptest): 1740 samples
Class 9 (DoS Slowhttptest - Attempted): 3368 samples
Class 10 (DoS Slowloris): 3859 samples
Class 11 (DoS Slowloris - Attempted): 1847 samples
Class 12 (FTP-Patator): 3972 samples
Class 13 (FTP-Patator - Attempted): 12 samples
Class 14 (Heartbleed): 11 samples
Class 15 (Infiltration): 36 samples
Class 16 (Infiltration - Attempted): 45 samples
Class 17 (Infiltration - Portscan): 68620 samples
Class 18 (Portscan): 159059 samples
Class 19 (SSH-Patator): 2961 samples
Class 20 (SSH-Patator - Attempted): 27 samples
Class 21 (Web Attack - Brute Force): 73 samples
Class 22 (Web Attack - Brute Force - Attempted): 1292 samples
Class 23 (Web Attack - SQL Injec

In [33]:
# First split: separate test set (80% train+val, 20% test)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Second split: separate train and validation (64% train, 16% val, 20% test)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

# Convert to float32 for TabNet
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

print("\nData types after conversion:")
print(f"X_train dtype: {X_train.dtypes.iloc[0]}")
print("Data split completed!")

Training set: (1334220, 58)
Validation set: (333556, 58)
Test set: (416945, 58)

Data types after conversion:
X_train dtype: float32
Data split completed!


In [34]:
# Initialize TabNet
clf = TabNetClassifier(
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type="sparsemax",
    scheduler_params={"step_size":10, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    device_name=DEVICE,
    verbose=1
)

/usr/local/lib/python3.10/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [35]:
print("Starting TabNet model training...")
print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Features: {X_train.shape[1]}")


Starting TabNet model training...
Training samples: 1334220
Validation samples: 333556
Features: 58


##  Training the model

In [36]:
# Correct training with validation data
clf.fit(
    X_train=X_train.values, y_train=y_train,
    eval_set=[(X_val.values, y_val)],
    patience=2, 
    num_workers=3,
    max_epochs=20,
    weights=1
)

epoch 0  | loss: 0.26409 | val_0_accuracy: 0.94764 |  0:02:48s
epoch 1  | loss: 0.08498 | val_0_accuracy: 0.97156 |  0:05:37s
epoch 2  | loss: 0.07388 | val_0_accuracy: 0.96516 |  0:08:16s
epoch 3  | loss: 0.06816 | val_0_accuracy: 0.97689 |  0:10:28s
epoch 4  | loss: 0.06266 | val_0_accuracy: 0.97799 |  0:12:54s
epoch 5  | loss: 0.06279 | val_0_accuracy: 0.98316 |  0:15:15s
epoch 6  | loss: 0.059   | val_0_accuracy: 0.97795 |  0:17:35s
epoch 7  | loss: 0.0576  | val_0_accuracy: 0.98197 |  0:20:08s

Early stopping occurred at epoch 7 with best_epoch = 5 and best_val_0_accuracy = 0.98316


/usr/local/lib/python3.10/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


## Evaluation

In [37]:
# Make predictions
y_pred = clf.predict(X_test.values)
y_pred_proba = clf.predict_proba(X_test.values)

In [38]:
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision_weighted = precision_score(y_test, y_pred, average='weighted')
recall_weighted = recall_score(y_test, y_pred, average='weighted')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

/usr/local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [39]:
# Multi-class AUC
auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')

# Results summary
results = {
    "Accuracy": f"{accuracy:.4f}",
    "Precision": f"{precision_weighted:.4f}",
    "Recall": f"{recall_weighted:.4f}",
    "F1-Score": f"{f1_weighted:.4f}",
    "AUC": f"{auc:.4f}"
}


In [ ]:
print("Model Performance:")
for metric, value in results.items():
    print(f"  {metric}: {value}")


### Model Performance  

In [40]:
for metric, value in results.items():
    print(f"  {metric}: {value}")

# Compare against success criteria
print(f"\nSuccess Criteria Assessment:")
criteria_check = {
    "Accuracy > 95%": accuracy > 0.95,
    "Precision > 90%": precision_weighted > 0.90,
    "Recall > 85%": recall_weighted > 0.85,
    "F1-Score > 90%": f1_weighted > 0.90
}

for criterion, passed in criteria_check.items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {criterion}: {status}")

# Feature importance analysis
feature_importance = clf.feature_importances_
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print(f"\nTop 10 Most Important Features:")
for i, (_, row) in enumerate(importance_df.head(10).iterrows(), 1):
    print(f"  {i}. {row['feature']}: {row['importance']:.4f}")


  Accuracy: 0.9834
  Precision: 0.9881
  Recall: 0.9834
  F1-Score: 0.9854
  AUC: 0.9994

Success Criteria Assessment:
  Accuracy > 95%: ✓ PASS
  Precision > 90%: ✓ PASS
  Recall > 85%: ✓ PASS
  F1-Score > 90%: ✓ PASS

Top 10 Most Important Features:
  1. ICMP Code: 0.3617
  2. Flow IAT Max: 0.1215
  3. SYN Flag Count: 0.1082
  4. Attempted Category: 0.0970
  5. Bwd IAT Mean: 0.0903
  6. Fwd Packet Length Mean: 0.0570
  7. Flow Packets/s: 0.0299
  8. Flow Duration: 0.0265
  9. Flow IAT Std: 0.0222
  10. Fwd Header Length: 0.0192


# Load model and test 

In [19]:
# define new model with basic parameters and load state dict weights
loaded_clf = TabNetClassifier()
loaded_clf.load_model("model.zip")

c:\Users\YOGA\Desktop\master s4 PFE\project\IDS-MAS\.venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [42]:
# Example with sample data
df = dd.read_csv('data/02-14-2018.csv', assume_missing=True)
dfc = df.compute()
dfc.head(10)

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.0,0.0,14/02/2018 08:31:01,112641719.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,56320859.5,139.300036,56320958.0,56320761.0,Benign
1,0.0,0.0,14/02/2018 08:33:50,112641466.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,56320733.0,114.551299,56320814.0,56320652.0,Benign
2,0.0,0.0,14/02/2018 08:36:39,112638623.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,56319311.5,301.934596,56319525.0,56319098.0,Benign
3,22.0,6.0,14/02/2018 08:40:13,6453966.0,15.0,10.0,1239.0,2273.0,744.0,0.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign
4,22.0,6.0,14/02/2018 08:40:23,8804066.0,14.0,11.0,1143.0,2209.0,744.0,0.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign
5,22.0,6.0,14/02/2018 08:40:31,6989341.0,16.0,12.0,1239.0,2273.0,744.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign
6,0.0,0.0,14/02/2018 08:39:28,112640480.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,56320240.0,203.646753,56320384.0,56320096.0,Benign
7,0.0,0.0,14/02/2018 08:42:17,112641244.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,56320622.0,62.225397,56320666.0,56320578.0,Benign
8,80.0,6.0,14/02/2018 08:47:14,476513.0,5.0,3.0,211.0,463.0,211.0,0.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign
9,80.0,6.0,14/02/2018 08:47:15,475048.0,5.0,3.0,220.0,472.0,220.0,0.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign


In [45]:
print(dfc.columns.tolist())


['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd By

In [ ]:
wanted_cols = [
    'Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet',
    'Total Length of Fwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min',
    'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max',
    'Bwd Packet Length Min', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean',
    'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Mean', 'Fwd IAT Std',
    'Bwd IAT Mean', 'Bwd IAT Std', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags',
    'Bwd URG Flags', 'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length',
    'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Min', 'FIN Flag Count',
    'SYN Flag Count', 'RST Flag Count', 'CWR Flag Count', 'ECE Flag Count',
    'Down/Up Ratio', 'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg',
    'Bwd Bytes/Bulk Avg', 'Bwd Bulk Rate Avg', 'Subflow Fwd Packets',
    'Subflow Bwd Packets', 'FWD Init Win Bytes', 'Bwd Init Win Bytes',
    'Fwd Act Data Pkts', 'Fwd Seg Size Min', 'Active Mean', 'Active Std',
    'Active Max', 'Active Min', 'Idle Std', 'ICMP Code', 'ICMP Type',
    'Total TCP Flow Time', 'Attempted Category', 'Label'
]

existing_cols = [c for c in wanted_cols if c in dfc.columns]
missing_cols = [c for c in wanted_cols if c not in dfc.columns]

dfc = dfc.drop(missing_cols,axis=1,)


KeyError: "['Src Port', 'Total Fwd Packet', 'Total Length of Fwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Flow Bytes/s', 'Flow Packets/s', 'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Min', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'CWR Flag Count', 'ECE Flag Count', 'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg', 'Bwd Bytes/Bulk Avg', 'Bwd Bulk Rate Avg', 'Subflow Fwd Packets', 'Subflow Bwd Packets', 'FWD Init Win Bytes', 'Bwd Init Win Bytes', 'ICMP Code', 'ICMP Type', 'Total TCP Flow Time', 'Attempted Category'] not found in axis"

In [41]:
# Separate features and target
X = dfc.drop('Label', axis=1)
y = dfc['Label']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {X.columns.tolist()}")
# Check for categorical columns
categorical_columns = X.select_dtypes(include=['object']).columns
print(f"Categorical columns: {categorical_columns.tolist()}")

# Encode categorical variables if any
for col in categorical_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    print(f"Encoded {col}")

# Encode target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\nClasses: {label_encoder.classes_}")
print(f"Number of classes: {len(label_encoder.classes_)}")
# Display class distribution after encoding
unique, counts = np.unique(y_encoded, return_counts=True)
for i, (class_idx, count) in enumerate(zip(unique, counts)):
    print(f"Class {class_idx} ({label_encoder.classes_[class_idx]}): {count} samples")

# First split: separate test set (80% train+val, 20% test)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Second split: separate train and validation (64% train, 16% val, 20% test)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

# Convert to float32 for TabNet
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

print("\nData types after conversion:")
print(f"X_train dtype: {X_train.dtypes.iloc[0]}")
print("Data split completed!")

KeyError: "['Label'] not found in axis"

In [17]:
# Make predictions
y_pred = loaded_clf.predict(X_test.values)
y_pred_proba = loaded_clf.predict_proba(X_test.values)

RuntimeError: running_mean should contain 78 elements not 58

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision_weighted = precision_score(y_test, y_pred, average='weighted')
recall_weighted = recall_score(y_test, y_pred, average='weighted')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

/usr/local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
# Multi-class AUC
auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')

# Results summary
results = {
    "Accuracy": f"{accuracy:.4f}",
    "Precision": f"{precision_weighted:.4f}",
    "Recall": f"{recall_weighted:.4f}",
    "F1-Score": f"{f1_weighted:.4f}",
    "AUC": f"{auc:.4f}"
}


In [ ]:
print("Model Performance:")
for metric, value in results.items():
    print(f"  {metric}: {value}")


### Model Performance  

In [ ]:
for metric, value in results.items():
    print(f"  {metric}: {value}")

# Compare against success criteria
print(f"\nSuccess Criteria Assessment:")
criteria_check = {
    "Accuracy > 95%": accuracy > 0.95,
    "Precision > 90%": precision_weighted > 0.90,
    "Recall > 85%": recall_weighted > 0.85,
    "F1-Score > 90%": f1_weighted > 0.90
}

for criterion, passed in criteria_check.items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {criterion}: {status}")

# Feature importance analysis
feature_importance = clf.feature_importances_
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print(f"\nTop 10 Most Important Features:")
for i, (_, row) in enumerate(importance_df.head(10).iterrows(), 1):
    print(f"  {i}. {row['feature']}: {row['importance']:.4f}")


  Accuracy: 0.9834
  Precision: 0.9881
  Recall: 0.9834
  F1-Score: 0.9854
  AUC: 0.9994

Success Criteria Assessment:
  Accuracy > 95%: ✓ PASS
  Precision > 90%: ✓ PASS
  Recall > 85%: ✓ PASS
  F1-Score > 90%: ✓ PASS

Top 10 Most Important Features:
  1. ICMP Code: 0.3617
  2. Flow IAT Max: 0.1215
  3. SYN Flag Count: 0.1082
  4. Attempted Category: 0.0970
  5. Bwd IAT Mean: 0.0903
  6. Fwd Packet Length Mean: 0.0570
  7. Flow Packets/s: 0.0299
  8. Flow Duration: 0.0265
  9. Flow IAT Std: 0.0222
  10. Fwd Header Length: 0.0192


In [41]:
# save tabnet model
saving_path_name = "./tabnet_model_test_1"
saved_filepath = clf.save_model(saving_path_name)

# define new model with basic parameters and load state dict weights
loaded_clf = TabNetClassifier()
loaded_clf.load_model(saved_filepath)

Successfully saved model at ./tabnet_model_test_1.zip


/usr/local/lib/python3.10/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
